# W13 — Hyperparameter Optimisation (Grid Search + Grouped K-Fold), per Configuration

**Purpose.** The hyperparameters used so far were selected in W05/W07 on the BI data that contained target leakage, and with a cross-validation that scored complete run-to-failure trajectories (RUL ≈ 0 at the last window, hence CV RMSE of 2–4). This notebook re-selects them on the corrected data with the evaluation protocol of W12. **Run it before W12**: W12 reads the selected hyperparameters from Drive.

**Design.**
- Grid search with grouped K-fold cross-validation on the training engines of the fleet set in Section 0 (FD001 + FD003, 200 engines; 5 folds for XGBoost, 3 for the BiLSTM, as in the original design). Each held-out engine is truncated at 3 random points (10–95% of its life) and the RUL is predicted at those points. The truncation points use a **different random seed** from W12.
- One optimisation **per feature configuration** (sensors only; + 13 operational BI; + 18 BI), so that every configuration receives the same tuning effort.
- Feature selection and scaling are fitted inside each fold (correlation-based selection for XGBoost, stratified AFICv for the BiLSTM). Asset history goes to the ML branch only (W12, Section 4.1), so the BiLSTM inputs of the 18-BI configuration are those of the 13-BI configuration: the BiLSTM is tuned for sensors only and for + 13 BI.
- The grids contain the default settings, which gives the before/after comparison of the article (default vs optimised).

| Branch | Grid | Combinations |
|---|---|---|
| XGBoost | trees {300, 500} × depth {4, 6, 8} × learning rate {0.05, 0.1} × window representation {statistics, flatten} | 24 |
| BiLSTM-DA | LSTM units {32, 64} × temporal attention {64, 128} × dense units {32, 64} × learning rate {0.0005, 0.001}; feature attention 32, dropout 0.3, batch 128 | 16 |

**Note for the article.** Hyperparameters are selected on the same engines that W12 then evaluates by cross-validation (with different truncation points). This mild selection effect is identical for all configurations; it is stated as a limitation.

Every evaluated (configuration, combination, fold) is cached on Drive, so an interrupted run resumes.

## 0. Setup and data (same code as W12)

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
from datetime import datetime

IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    # Fresh clone every run; cache and results live on Drive so finished work survives a disconnect
    if os.path.exists('/content/PhD-Project-'):
        !rm -rf /content/PhD-Project-
    !git clone https://github.com/f-khadija-benzine/PhD-Project-.git /content/PhD-Project-
    !pip install xgboost shap -q
    os.chdir('/content/PhD-Project-/src')
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/PhD-Project-')
    TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M')
    SAVE_DIR = Path(f'/content/drive/MyDrive/PhD_results/W13_{TIMESTAMP}')
    CACHE_ROOT = Path('/content/drive/MyDrive/PhD_results/W13_cache')
else:
    PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
    SAVE_DIR = None
    CACHE_ROOT = PROJECT_ROOT / 'results' / 'W13_cache'

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT))

CMAPSS_DIR = PROJECT_ROOT / 'data' / 'raw_data' / 'C_MAPSS'
BI_DIR     = PROJECT_ROOT / 'data' / 'raw_data' / 'BI_data'
ASSET_DIR  = BI_DIR / 'asset'
# ---- Fleet: FD001 alone, or FD001 + FD003 analysed as ONE fleet of 200 engines ----
# FD003: same single operating condition as FD001, two fault modes (HPC + fan).
# Unit ids of FD003 are offset by +1000. Set DATASETS = ['FD001'] to reproduce the FD001-only runs.
DATASETS = ['FD001', 'FD003']
for _ds in DATASETS:
    assert (CMAPSS_DIR / f'train_{_ds}.txt').exists(), f"train_{_ds}.txt missing in {CMAPSS_DIR}: upload it to the repo"
FLEET_TAG = '_'.join(DATASETS)
TAG_SUFFIX = '' if FLEET_TAG == 'FD001' else f'_{FLEET_TAG}'   # FD001 keeps its original paths and caches

RESULTS    = PROJECT_ROOT / 'results' / f'W13{TAG_SUFFIX}'
CACHE_ROOT = CACHE_ROOT.parent / f'{CACHE_ROOT.name}{TAG_SUFFIX}'
if SAVE_DIR is not None:
    SAVE_DIR = SAVE_DIR.parent / f'{SAVE_DIR.name}{TAG_SUFFIX}'
for d in (RESULTS, CACHE_ROOT):
    d.mkdir(parents=True, exist_ok=True)
if SAVE_DIR is not None:
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Fleet: {FLEET_TAG}\nProject root: {PROJECT_ROOT}\nResults: {RESULTS}\nCache: {CACHE_ROOT}\nDrive save: {SAVE_DIR}")

In [ ]:
# Regenerate both BI blocks from the generators in the repo, so data and code always match
for script in ['generate_bi_data.py', 'generate_bi_asset_data.py']:
    res = subprocess.run([sys.executable, str(PROJECT_ROOT / 'src' / script)],
                         capture_output=True, text=True, cwd=str(PROJECT_ROOT))
    print(f"--- {script}: {'OK' if res.returncode == 0 else 'FAILED'}")
    if res.returncode != 0:
        print(res.stderr[-2000:])
        raise RuntimeError(f"{script} failed")

In [ ]:
import importlib, time, json, io, contextlib, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from numpy.lib.stride_tricks import sliding_window_view
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
import shap

import ml_branch, bi_fusion, attention, feature_selection, feature_selection_aficv, generate_bi_asset_data as gba
for m in (ml_branch, bi_fusion, attention, feature_selection, feature_selection_aficv, gba):
    importlib.reload(m)
from ml_branch import MLBranch
from attention import build_dual_attention_bilstm
from feature_selection import BIAwareFeatureSelector
from feature_selection_aficv import AFICvFeatureSelector
from bi_fusion import CATEGORICAL_BI_VARS, CONTINUOUS_BI_VARS, BI_FEATURE_NAMES

# ---------------- Experiment configuration (fixed before running) ----------------
RHO = 0.4                      # asset-history link strength, chosen from W11 realism checks
FLEET = 2026                   # reference simulated fleet (= repo files)
SEEDS = [42, 43, 44]           # repeated CV (advisor point 4)
W = 30
RUL_MAX = 125
N_FOLDS = 5
N_CUTS = 3
INNER_VAL_FRAC = 0.2
CUT_SEED = 42                  # same truncation points as W11
ASSET_ROUTING = 'ml_only'      # static asset history goes to the ML branch only (see Section 4.1)

RF_PARAMS = dict(n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=2)   # benchmark, not tuned
BATCH, EPOCHS = 128, 100
DENSE_EPOCHS = 300           # dense fusion network (small, early-stopped; independent of QUICK)
CACHE_VERSION = 'v2'           # v2 = hyperparameters from W13 (v1 runs used the W05/W07 settings)
CORR_SEL = dict(variance_threshold=0.01, correlation_threshold=0.95)
AFICV_SEL = dict(base_learner='xgboost', n_folds=5, cumulative_threshold=0.90)

# QUICK = True: smoke test (tiny models, 1 seed). Use False for reported results.
QUICK = False
if QUICK:
    SEEDS = [42]; EPOCHS = 2
    RF_PARAMS = dict(n_estimators=30, max_depth=8, min_samples_split=5, min_samples_leaf=2)

GPU = tf.config.list_physical_devices('GPU')
XGB_EXTRA = {'device': 'cuda', 'tree_method': 'hist'} if GPU else {}
CACHE_DIR = CACHE_ROOT / ('quick' if QUICK else 'full')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"QUICK={QUICK} | seeds={SEEDS} | epochs={EPOCHS} | GPU={bool(GPU)} | cache={CACHE_DIR}")

In [ ]:
cols = ['unit', 'cycle'] + [f'setting_{i}' for i in range(1, 4)] + [f'sensor_{i}' for i in range(1, 22)]
def _load_train(ds):
    t = pd.read_csv(CMAPSS_DIR / f'train_{ds}.txt', sep=r'\s+', header=None, names=cols, engine='python')
    t['unit'] += gba.UNIT_OFFSET[ds]
    t['subset'] = ds
    return t
train = pd.concat([_load_train(ds) for ds in DATASETS], ignore_index=True)
train['total_life'] = train.groupby('unit')['cycle'].transform('max')
train['rul'] = (train['total_life'] - train['cycle']).clip(upper=RUL_MAX)
UNITS = np.sort(train['unit'].unique())
LIFE = train.groupby('unit')['total_life'].first()

SENSOR_COLS = [c for c in train.columns if c.startswith('sensor_')]
SETTING_COLS = [c for c in train.columns if c.startswith('setting_')]

# Operational BI (13 variables)
bi_op = pd.read_csv(BI_DIR / f'BI_{FLEET_TAG}_train.csv').rename(columns={'unit_id': 'unit'})
OP_VARS = [c for c in BI_FEATURE_NAMES if c in bi_op.columns]
OP_CAT = [c for c in CATEGORICAL_BI_VARS if c in OP_VARS]
OP_CONT = [c for c in CONTINUOUS_BI_VARS if c in OP_VARS]

# Asset-history BI (5 variables), reference fleet, rho fixed above
asset = pd.read_csv(ASSET_DIR / f"BI_{FLEET_TAG}_asset_{gba.rho_tag(RHO)}.csv").rename(columns={'unit_id': 'unit'})
ASSET_VARS = gba.ASSET_VARS
ASSET_CONT = ['prior_unplanned_removals', 'hours_since_overhaul_entry']
BATCH_LEVELS = [f"B{i}" for i in range(1, gba.N_BATCHES + 1)]

def build_table(op_df, asset_df):
    # One table with sensors + all BI, categorical variables one-hot encoded
    df = train[['unit', 'cycle', 'rul'] + SETTING_COLS + SENSOR_COLS].merge(
        op_df[['unit', 'cycle'] + OP_VARS], on=['unit', 'cycle'], how='left')
    for c in OP_CAT:
        df[c] = pd.Categorical(df[c].astype(int), categories=sorted(bi_op[c].unique()))
    df = pd.get_dummies(df, columns=OP_CAT, dtype=float)
    a = asset_df[['unit'] + ASSET_VARS].copy()
    a['component_batch'] = pd.Categorical(a['component_batch'], categories=BATCH_LEVELS)
    df = df.merge(pd.get_dummies(a, columns=['component_batch'], dtype=float), on='unit', how='left')
    return df.sort_values(['unit', 'cycle']).reset_index(drop=True)

# Falsification controls (same methods as W10/W11):
#  - operational BI: each variable permuted independently across all rows
#  - asset history: full records permuted across engines (stays constant within an engine)
op_shuf = bi_op.copy()
r_op = np.random.RandomState(42)
for c in OP_VARS:
    op_shuf[c] = r_op.permutation(op_shuf[c].values)
asset_shuf = asset.sort_values('unit').reset_index(drop=True).copy()
perm = np.random.default_rng(FLEET + 1000).permutation(len(asset_shuf))
asset_shuf[ASSET_VARS] = asset_shuf[ASSET_VARS].values[perm]

DATA = {'real': build_table(bi_op, asset),
        'asset_shuffled': build_table(bi_op, asset_shuf),
        'all_shuffled': build_table(op_shuf, asset_shuf)}
DF = DATA['real']
assert all(list(d.columns) == list(DF.columns) for d in DATA.values())

OP_COLS = [c for c in DF.columns if c in OP_CONT or c in OP_VARS or c.startswith(tuple(f"{v}_" for v in OP_CAT))]
ASSET_COLS = [c for c in DF.columns if c.startswith(tuple(ASSET_VARS))]
FEATURE_SETS = {
    'S':        {'bi': [],                    'data': 'real'},            # sensors + settings only
    'S+Op':     {'bi': OP_COLS,               'data': 'real'},            # + 13 operational BI (article's original BI)
    'S+Op+A':   {'bi': OP_COLS + ASSET_COLS,  'data': 'real'},            # + 5 asset-history BI = 18 BI variables
    'S+Op+A~':  {'bi': OP_COLS + ASSET_COLS,  'data': 'asset_shuffled'},  # control: asset history shuffled across engines
    'S+Op~+A~': {'bi': OP_COLS + ASSET_COLS,  'data': 'all_shuffled'},    # control: all 18 BI shuffled
}
MAIN_FSETS = ['S', 'S+Op', 'S+Op+A']
CONT_COLS = SENSOR_COLS + SETTING_COLS + OP_CONT + ASSET_CONT
GROUP_OF = {**{c: 'sensor' for c in SENSOR_COLS + SETTING_COLS},
            **{c: 'operational BI' for c in OP_COLS}, **{c: 'asset BI' for c in ASSET_COLS}}

print(f"Fleet {FLEET_TAG}: " + ", ".join(f"{s} {n} engines" for s, n in train.groupby("subset").unit.nunique().items()))
print(f"Engines: {len(UNITS)}, rows: {len(DF)}, life [{LIFE.min()}, {LIFE.max()}]")
print(f"Columns: {len(SENSOR_COLS)} sensors + {len(SETTING_COLS)} settings + {len(OP_COLS)} operational BI "
      f"(13 variables) + {len(ASSET_COLS)} asset BI (5 variables)")

In [ ]:
folds = list(GroupKFold(n_splits=N_FOLDS).split(UNITS, groups=UNITS))

rng = np.random.default_rng(CUT_SEED)
cut_rows = []
for u in UNITS:
    life = int(LIFE[u])
    lo, hi = max(int(0.10 * life), 1), max(int(0.95 * life), 2)
    for k, c in enumerate(np.sort(rng.integers(lo, hi, N_CUTS))):
        cut_rows.append({'unit': u, 'cut_id': k, 'cut_cycle': int(c), 'true_rul': min(life - int(c), RUL_MAX)})
CUTS = pd.DataFrame(cut_rows)

def inner_split(train_units, seed):
    r = np.random.default_rng(seed)
    val = r.choice(train_units, size=int(round(INNER_VAL_FRAC * len(train_units))), replace=False)
    return np.setdiff1d(train_units, val), np.sort(val)

assert all(len(set(UNITS[tr]) & set(UNITS[te])) == 0 for tr, te in folds)
print(f"{N_FOLDS} folds × {[len(te) for _, te in folds]} test engines; "
      f"{len(CUTS)} truncation points, true RUL {CUTS.true_rul.min()}–{CUTS.true_rul.max()}")

In [ ]:
STAT_SUFFIX = ('_mean', '_std', '_min', '_max', '_last', '_trend')
def is_static(c): return c.startswith(tuple(ASSET_VARS))
def original_name(flat):
    for s in STAT_SUFFIX:
        if flat.endswith(s):
            return flat[: -len(s)]
    return flat

def quiet(fn, *args, **kw):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args, **kw)

def scale_fold(train_units, fset):
    d = DATA[FEATURE_SETS[fset]['data']].copy()
    sc = MinMaxScaler().fit(d.loc[d.unit.isin(train_units), CONT_COLS])
    d[CONT_COLS] = sc.transform(d[CONT_COLS])
    return d

def select_features(d, train_units, fset, method, exclude_asset=False):
    # Returns the selected column list, fitted on training engines only
    tr = d[d.unit.isin(train_units)]
    bi = FEATURE_SETS[fset]['bi']
    if exclude_asset:
        bi = [c for c in bi if c not in ASSET_COLS]
    if method == 'correlation':
        sel = BIAwareFeatureSelector(**CORR_SEL)
        return quiet(sel.select_features, data=tr, sensor_cols=SENSOR_COLS, bi_cols=bi,
                     setting_cols=SETTING_COLS, exclude_cols=['unit', 'cycle', 'rul'])
    if method == 'aficv':
        sel = AFICvFeatureSelector(**AFICV_SEL)
        if bi:
            return quiet(sel.select_features_stratified, data=tr, sensor_cols=SENSOR_COLS, bi_cols=bi,
                         setting_cols=SETTING_COLS, target_col='rul', group_col='unit')
        return quiet(sel.select_features, data=tr, feature_cols=SENSOR_COLS + SETTING_COLS,
                     target_col='rul', group_col='unit')
    raise ValueError(method)

def windows_train(d, units, feats):
    # PAD=False as in the article: windows ending at cycles W..T of each engine
    X, y, owner = [], [], []
    for u in units:
        g = d[d.unit == u]
        v = g[feats].to_numpy(np.float32)
        if len(v) < W:
            continue
        X.append(sliding_window_view(v, (W, v.shape[1]))[:, 0])
        y.append(g['rul'].to_numpy(np.float32)[W - 1:])
        owner += [u] * (len(v) - W + 1)
    return np.concatenate(X), np.concatenate(y), np.asarray(owner)

def windows_cuts(d, units, feats):
    # Last window before each truncation point (zero-padded if the engine is shorter than W)
    X, keys = [], []
    for u in units:
        v_full = d[d.unit == u][feats].to_numpy(np.float32)
        for _, c in CUTS[CUTS.unit == u].iterrows():
            v = v_full[:c.cut_cycle]
            if len(v) < W:
                v = np.vstack([np.zeros((W - len(v), len(feats)), np.float32), v])
            X.append(v[-W:]); keys.append((u, int(c.cut_id)))
    return np.asarray(X), keys

def static_rows(d, owners, static_cols):
    return d.groupby('unit')[static_cols].first().loc[list(owners)].to_numpy(np.float32)

def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

## 1. Tuning protocol

In [ ]:
TUNE_CUT_SEED = 7                      # different truncation points from W12 (seed 42)
TUNE_FOLDS = {'ml': 5, 'dl': 3}
TUNE_SEED = 42
CONFIGS_ML = ['S', 'S+Op', 'S+Op+A']
CONFIGS_DL = ['S', 'S+Op']            # S+Op+A uses the S+Op BiLSTM settings (no asset history in the DL branch)

ML_GRID = {'n_estimators': [300, 500], 'max_depth': [4, 6, 8], 'learning_rate': [0.05, 0.1],
           'flatten': ['statistics', 'flatten']}
DL_GRID = {'lstm_units': [32, 64], 'temporal_attention_dim': [64, 128], 'dense_units': [32, 64],
           'learning_rate': [0.0005, 0.001]}
DL_FIXED = {'feature_attention_dim': 32, 'dropout_rate': 0.3}
ML_DEFAULT = {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'flatten': 'statistics'}
DL_DEFAULT = {'lstm_units': 64, 'temporal_attention_dim': 64, 'dense_units': 32, 'learning_rate': 0.001}

if QUICK:
    ML_GRID = {'n_estimators': [50], 'max_depth': [4], 'learning_rate': [0.05, 0.1], 'flatten': ['statistics', 'flatten']}
    DL_GRID = {'lstm_units': [32], 'temporal_attention_dim': [64], 'dense_units': [32], 'learning_rate': [0.0005, 0.001]}
    ML_DEFAULT = {'n_estimators': 50, 'max_depth': 4, 'learning_rate': 0.05, 'flatten': 'statistics'}
    DL_DEFAULT = {'lstm_units': 32, 'temporal_attention_dim': 64, 'dense_units': 32, 'learning_rate': 0.001}

def combos(grid):
    keys = list(grid)
    return [dict(zip(keys, v)) for v in itertools.product(*grid.values())]
ML_COMBOS, DL_COMBOS = combos(ML_GRID), combos(DL_GRID)
assert ML_DEFAULT in ML_COMBOS and DL_DEFAULT in DL_COMBOS

# Truncation points for tuning (replace the W12 ones in the shared helper functions)
rng_t = np.random.default_rng(TUNE_CUT_SEED)
rows = []
for u in UNITS:
    life = int(LIFE[u]); lo, hi = max(int(0.10 * life), 1), max(int(0.95 * life), 2)
    for k, c in enumerate(np.sort(rng_t.integers(lo, hi, N_CUTS))):
        rows.append({'unit': u, 'cut_id': k, 'cut_cycle': int(c), 'true_rul': min(life - int(c), RUL_MAX)})
CUTS = pd.DataFrame(rows)
TFOLDS = {b: list(GroupKFold(n_splits=n).split(UNITS, groups=UNITS)) for b, n in TUNE_FOLDS.items()}
TUNE_DIR = CACHE_DIR / 'tuning'; TUNE_DIR.mkdir(parents=True, exist_ok=True)
print(f"ML: {len(ML_COMBOS)} combinations × {TUNE_FOLDS['ml']} folds × {len(CONFIGS_ML)} configs = "
      f"{len(ML_COMBOS) * TUNE_FOLDS['ml'] * len(CONFIGS_ML)} fits")
print(f"DL: {len(DL_COMBOS)} combinations × {TUNE_FOLDS['dl']} folds × {len(CONFIGS_DL)} configs = "
      f"{len(DL_COMBOS) * TUNE_FOLDS['dl'] * len(CONFIGS_DL)} fits")

In [ ]:
_SEL, _SCALED = {}, {}
def fold_data(branch, fset, k):
    # Scaled table and selected features for one tuning fold (computed once, reused by every combination)
    key = (branch, fset, k)
    if key not in _SEL:
        tr_u = UNITS[TFOLDS[branch][k][0]]
        d = scale_fold(tr_u, fset)
        if branch == 'ml':
            f = select_features(d, tr_u, fset, 'correlation')
        else:
            f = select_features(d, tr_u, fset, 'aficv', exclude_asset=True)
        _SCALED[key], _SEL[key] = d, f
    return _SCALED[key], _SEL[key]

def eval_ml(fset, p, k):
    d, feats = fold_data('ml', fset, k)
    tr_u, te_u = UNITS[TFOLDS['ml'][k][0]], UNITS[TFOLDS['ml'][k][1]]
    dyn = [c for c in feats if not is_static(c)]; stat = [c for c in feats if is_static(c)]
    Xtr, ytr, otr = windows_train(d, tr_u, dyn)
    Xte, keys = windows_cuts(d, te_u, dyn)
    ml = MLBranch(model_type='xgboost', flatten_strategy=p['flatten'], random_state=TUNE_SEED,
                  n_estimators=p['n_estimators'], max_depth=p['max_depth'], learning_rate=p['learning_rate'], **XGB_EXTRA)
    Ftr, Fte = ml._flatten(Xtr, dyn), ml._flatten(Xte, dyn)
    if stat:
        Ftr = np.hstack([Ftr, static_rows(d, otr, stat)])
        Fte = np.hstack([Fte, static_rows(d, [u for u, _ in keys], stat)])
    t0 = time.perf_counter(); ml.model.fit(Ftr, ytr); t = time.perf_counter() - t0
    y = CUTS.set_index(['unit', 'cut_id']).loc[keys, 'true_rul'].to_numpy(float)
    pr = ml.model.predict(Fte)
    return {'rmse': rmse(y, pr), 'score': nasa(y, pr), 'time': t}

def eval_dl(fset, p, k):
    d, feats = fold_data('dl', fset, k)
    tr_u, te_u = UNITS[TFOLDS['dl'][k][0]], UNITS[TFOLDS['dl'][k][1]]
    fit_u, val_u = inner_split(tr_u, TUNE_SEED * 100 + k)
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(TUNE_SEED)
    Xtr, ytr, _ = windows_train(d, fit_u, feats); Xva, yva, _ = windows_train(d, val_u, feats)
    Xte, keys = windows_cuts(d, te_u, feats)
    model, _ = build_dual_attention_bilstm(window_size=W, n_features=len(feats), **DL_FIXED, **p)
    cb = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
          tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)]
    t0 = time.perf_counter()
    h = model.fit(Xtr, ytr, epochs=EPOCHS, batch_size=BATCH, validation_data=(Xva, yva), callbacks=cb, verbose=0)
    t = time.perf_counter() - t0
    y = CUTS.set_index(['unit', 'cut_id']).loc[keys, 'true_rul'].to_numpy(float)
    pr = model.predict(Xte, batch_size=256, verbose=0).ravel()
    return {'rmse': rmse(y, pr), 'score': nasa(y, pr), 'time': t, 'epochs': len(h.history['loss'])}

def nasa(y, p):
    dd = np.asarray(p) - np.asarray(y)
    return float(np.sum(np.where(dd < 0, np.exp(-dd / 13) - 1, np.exp(dd / 10) - 1)))

def cached_eval(branch, fset, cid, p, k):
    path = TUNE_DIR / f"{branch}_{fset.replace('+', 'p')}_c{cid}_f{k}.json"
    if path.exists():
        return json.loads(path.read_text())
    r = (eval_ml if branch == 'ml' else eval_dl)(fset, p, k)
    r.update({'branch': branch, 'fset': fset, 'combo': cid, 'fold': k, **{f"p_{a}": b for a, b in p.items()}})
    path.write_text(json.dumps(r))
    return r
print("Tuning functions ready ✓")

## 2. Grid search — XGBoost branch

In [ ]:
t0 = time.time(); R = []
for fset in CONFIGS_ML:
    for cid, p in enumerate(ML_COMBOS):
        R += [cached_eval('ml', fset, cid, p, k) for k in range(TUNE_FOLDS['ml'])]
    print(f"  ML {fset:<7} done ({(time.time() - t0) / 60:.1f} min)")
ML_RES = pd.DataFrame(R)
ML_RES.to_csv(RESULTS / 'w13_ml_grid.csv', index=False)

## 3. Grid search — BiLSTM-DA branch

In [ ]:
t0 = time.time(); R = []
for fset in CONFIGS_DL:
    for cid, p in enumerate(DL_COMBOS):
        R += [cached_eval('dl', fset, cid, p, k) for k in range(TUNE_FOLDS['dl'])]
        print(f"  DL {fset:<5} combo {cid + 1}/{len(DL_COMBOS)} ({(time.time() - t0) / 60:.1f} min)")
DL_RES = pd.DataFrame(R)
DL_RES.to_csv(RESULTS / 'w13_dl_grid.csv', index=False)

## 4. Selected hyperparameters and default vs optimised

For each branch and configuration: mean RMSE over the folds for every combination; the combination with the lowest mean RMSE is selected. The default combination is reported for the before/after comparison.

In [ ]:
def summarise(res, combos_list, default, branch):
    pcols = [c for c in res.columns if c.startswith('p_')]
    g = res.groupby(['fset', 'combo']).agg(rmse=('rmse', 'mean'), rmse_sd=('rmse', 'std'),
                                           score=('score', 'mean'), time=('time', 'mean')).reset_index()
    def_id = combos_list.index(default)
    rows, best = [], {}
    for fset, x in g.groupby('fset'):
        b = x.loc[x.rmse.idxmin()]; dft = x[x.combo == def_id].iloc[0]
        best[fset] = combos_list[int(b.combo)]
        rows.append({'branch': branch, 'Features': fset,
                     'RMSE default': dft.rmse, 'RMSE optimised': b.rmse, 'RMSE SD (folds)': b.rmse_sd,
                     'RMSE gain %': 100 * (dft.rmse - b.rmse) / dft.rmse,
                     'Score default': dft.score, 'Score optimised': b.score,
                     'Score gain %': 100 * (dft.score - b.score) / dft.score,
                     'selected': combos_list[int(b.combo)]})
    return pd.DataFrame(rows), best, g

ml_tab, ml_best, ml_g = summarise(ML_RES, ML_COMBOS, ML_DEFAULT, 'XGBoost')
dl_tab, dl_best, dl_g = summarise(DL_RES, DL_COMBOS, DL_DEFAULT, 'BiLSTM-DA')
dl_best['S+Op+A'] = dl_best['S+Op']          # no asset history in the DL branch

OPT = pd.concat([ml_tab, dl_tab], ignore_index=True)
OPT.to_csv(RESULTS / 'w13_default_vs_optimised.csv', index=False)
display(OPT.drop(columns='selected').round(2))
for _, r in OPT.iterrows():
    print(f"{r.branch:<10} {r.Features:<7} → {r.selected}")

BEST = {f: {'ml': ml_best[f], 'dl': {**dl_best[f], **DL_FIXED}} for f in CONFIGS_ML}
BEST_PATH_LOCAL = RESULTS / f'W13_best_params{TAG_SUFFIX}.json'
BEST_PATH_LOCAL.write_text(json.dumps({'quick': QUICK, 'params': BEST}, indent=2))
if IN_COLAB:
    drive_path = Path(f'/content/drive/MyDrive/PhD_results/W13_best_params{TAG_SUFFIX}.json')
    drive_path.write_text(json.dumps({'quick': QUICK, 'params': BEST}, indent=2))
    print(f"\nSaved for W12: {drive_path}")
print(f"Saved: {BEST_PATH_LOCAL}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, (g, name) in zip(axes, [(ml_g, 'XGBoost'), (dl_g, 'BiLSTM-DA')]):
    for fset, x in g.groupby('fset'):
        ax.plot(np.sort(x.rmse.values), marker='o', ms=3, label=fset)
    ax.set_xlabel('Combinations, sorted by CV RMSE'); ax.set_ylabel('Mean CV RMSE (cycles)')
    ax.set_title(f'{name}: grid-search results per configuration'); ax.legend()
plt.tight_layout(); plt.savefig(RESULTS / 'fig_w13_grid.png', dpi=150); plt.show()

if SAVE_DIR is not None:
    for f in RESULTS.iterdir():
        if f.is_file() and f.name.startswith(('w13', 'W13', 'fig_w13')):
            shutil.copy(f, SAVE_DIR / f.name)
    print(f"Results copied to {SAVE_DIR}")

## 5. For the article (Section 5.2)

- Report the default vs optimised table per branch and configuration (RMSE and NASA score, with the SD over folds).
- State that tuning used grouped K-fold with truncation, identical effort per configuration, grids containing the defaults, and that the same engines are later used for the main cross-validation (different truncation points).
- The GA comparison of the previous version is not repeated.